# Presentation figures

Redraws the plots of the capstone presentation for a mixed audience: one message per plot,
direct labels, large fonts, slide-sized canvases. **No new analysis:** every number comes from
the existing exports (`data/models/`, `data/metrics/`, `data/risk_classification/`) or from
`data/smard.csv`, loaded as in `team-EDA.ipynb` §1.

- Output: `presentation/figures/*.png` (committed) and `presentation/slide_numbers.csv`, every
  number the slides quote, with its source.
- Picks: the headline slides show the random forest hybrid (best single model on the test year);
  the extremes, risk and money figures keep the team's adopted picks (specs 09-11).
- Colours: the app palette (`streamlit/viz_helpers.py`, chart-style skill) and the model colours
  of the visualization notebooks.

## 1 Setup
### 1.1 Configuration

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXPORT_ENABLED = True
DPI = 200            # 1 figure inch = 100 slide px, saved at 2x
HEADLINE = ("random_forest_hybrid", "rolling")  # best single model on the test year
ENSEMBLE = ("ensemble_regime_weighted", "rolling")  # spec 09's ensemble pick
BIN_PICKS = {  # spec 09 rank 1 per bin (viz-01 §2); "overall" is the headline model instead
    "overall": "random_forest_hybrid",
    "ordinary": "linear_direct",
    "below_zero": "xgb_hybrid",
    "low_extreme": "random_forest_hybrid",
    "high_extreme": "xgb_hybrid",
}
RISK_PICKS = {"high": "xgb_hybrid", "low": "random_forest_hybrid"}  # spec 10 PICKS

# Slide look: background and ink match the deck
BG = "#FAF9F6"
INK = "#1C2430"
MUTED = "#5A6472"
LIGHT = "#C9CED6"
GRID = "#E3E6EA"

# App palette (streamlit/viz_helpers.py COLORS / SERIES_COLOR)
COLORS = {
    "grid_load": "#D55E00",
    "residual_load": "#1C1C1C",
    "renewables": "#56B4E9",
    "solar": "#E69F00",
    "high": "#D55E00",
    "low": "#0072B2",
}
# Model colours (viz-01/viz-03 STYLE)
MODEL_STYLE = {
    "lgbm_direct": {"label": "LightGBM direct", "color": "#2C6EBA"},
    "lgbm_hybrid": {"label": "LightGBM hybrid", "color": "#2F8F5B"},
    "xgb_direct": {"label": "XGBoost direct", "color": "#E95D0F"},
    "xgb_hybrid": {"label": "XGBoost hybrid", "color": "#B10F0F"},
    "linear_direct": {"label": "Ridge direct", "color": "#7A4FA3"},
    "random_forest_hybrid": {"label": "Random forest hybrid", "color": "#8C564B"},
    "ensemble_regime_weighted": {"label": "Ensemble (regime-weighted)", "color": "#C2387A"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A"},
    "actual": {"label": "Actual", "color": "#1C1C1C"},
}

plt.rcParams.update({
    "font.family": ["Segoe UI", "Arial", "DejaVu Sans"],
    "font.size": 20,
    "axes.labelsize": 20,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "axes.edgecolor": LIGHT,
    "axes.labelcolor": MUTED,
    "xtick.color": INK,
    "ytick.color": INK,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 1,
    "xtick.major.size": 0,
    "ytick.major.size": 0,
    "legend.frameon": False,
})


def new_fig(width_px, height_px, **kwargs):
    """Figure sized in slide pixels (1 inch = 100 px)."""
    return plt.subplots(figsize=(width_px / 100, height_px / 100), layout="constrained", **kwargs)


def thousands(ax, axis="y"):
    """Thousands separators on an axis."""
    target = ax.yaxis if axis == "y" else ax.xaxis
    target.set_major_formatter(lambda v, _: f"{v:,.0f}")


def save(fig, name):
    """Write a figure to presentation/figures/ if EXPORT_ENABLED."""
    if EXPORT_ENABLED:
        fig.savefig(FIG_DIR / f"{name}.png", dpi=DPI)
    plt.show()


FACTS = []


def fact(slide, name, value, source):
    """Record a number a slide quotes."""
    FACTS.append({"slide": slide, "name": name, "value": value, "source": source})
    return value

### 1.2 Loading

`smard.csv` exactly as `team-EDA.ipynb` §1.3-§1.4 (via the Streamlit port), the model exports as
the visualization notebooks read them.

In [ ]:
DATA_DIR = next(
    (folder / "data" for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(f"No data/ folder found in {Path.cwd()} or any parent folder.")
FIG_DIR = DATA_DIR.parent / "presentation" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = {
    "Wind Offshore": "wind_off",
    "Wind Onshore": "wind_on",
    "Solar": "solar",
    "Grid Load": "grid_load",
    "Residual Load": "residual_load",
    "Forecast Wind + Solar": "fc_gen_wind_solar",
    "Forecast Grid Load": "fc_grid_load",
    "Forecast Residual Load": "fc_residual_load",
    "Capacity Wind Offshore": "cap_wind_off",
    "Capacity Wind Onshore": "cap_wind_on",
    "Capacity Solar": "cap_solar",
}
raw = pd.read_csv(DATA_DIR / "smard.csv", delimiter=";", encoding="utf-8-sig").rename(columns=COLUMNS)
raw["timestamp"] = pd.to_datetime(raw["timestamp"], format="%Y-%m-%d %H:%M")
for col in COLUMNS.values():
    raw[col] = raw[col].str.replace(",", ".").astype(float)
time_series = raw.set_index("timestamp").sort_index()
time_series["renewables"] = time_series[["wind_on", "wind_off", "solar"]].sum(axis=1)
time_series["year"] = time_series.index.year
time_series["hour"] = time_series.index.hour
time_series["date"] = time_series.index.date
YEARS = sorted(int(y) for y in time_series["year"].unique())


def load(path):
    """One export as a DataFrame, timestamps parsed."""
    frame = pd.read_csv(DATA_DIR / path)
    if "timestamp" in frame.columns:
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], format="%Y-%m-%d %H:%M:%S")
    return frame


scoreboard = pd.concat([
    load("models/model_scoreboard.csv"),
    load("models/ensemble_scoreboard.csv").query("model.str.startswith('ensemble_')"),
], ignore_index=True)
hourly = pd.concat([
    load("models/model_forecast_errors_hourly.csv"),
    load("models/ensemble_forecast_errors_hourly.csv"),
], ignore_index=True)
smard = load("metrics/smard_forecast_errors_hourly.csv").set_index("timestamp")
cost = load("models/model_rebap_cost_hourly.csv")
risk = load("risk_classification/risk_labels_daily.csv")
model_risk = load("risk_classification/model_risk_labels_daily.csv")

VALUE = scoreboard.pivot_table(index=["model", "split_method"], columns="metric", values="value")
COUNT = scoreboard.pivot_table(index=["model", "split_method"], columns="metric", values="count")
SMARD_ROW = ("smard", "none")


def series_of(row):
    """Hourly forecast rows of one (model, split_method), indexed by timestamp."""
    model, split = row
    return hourly.query("model == @model and split_method == @split").set_index("timestamp")


head = series_of(HEADLINE)
TEST_HOURS = head.index
print(f"data {time_series.index.min():%Y-%m-%d} .. {time_series.index.max():%Y-%m-%d %H:%M}, "
      f"test {TEST_HOURS.min():%Y-%m-%d} .. {TEST_HOURS.max():%Y-%m-%d} ({len(TEST_HOURS):,} hours)")

### 1.3 Checks

The exports and `smard.csv` must come from the same snapshot, and the headline numbers recomputed
here must equal the scoreboard.

In [ ]:
assert time_series.index.max() == TEST_HOURS.max(), "smard.csv and the model exports differ: re-run"
abs_err = pd.DataFrame({
    "model": head["err_residual_load"].abs(),
    "smard": smard.loc[TEST_HOURS, "err_residual_load"].abs(),
})
skill = 100 * (1 - abs_err["model"].mean() / abs_err["smard"].mean())
common = VALUE.loc[HEADLINE, "skill_pct"]
assert abs(skill - common) < 0.2, f"recomputed skill {skill:.2f} vs scoreboard {common:.2f}"
print(f"headline skill: {common:.1f} % (scoreboard), {skill:.1f} % (all {len(abs_err):,} hours)")

## 2 Why: the grid
### 2.1 Two extreme episodes

The highest 24-hour mean and the longest run below zero (team-EDA §3.8), on one shared y-axis.

In [ ]:
residual = time_series["residual_load"]
roll24 = residual.rolling(24).mean()
peak_end = roll24.idxmax()
peak_start = residual.index[residual.index.get_loc(peak_end) - 23]

negative = residual < 0
run_id = (negative != negative.shift()).cumsum()
runs = residual[negative].groupby(run_id[negative]).agg(
    start=lambda s: s.index.min(), end=lambda s: s.index.max(), hours="size"
)
longest = runs.sort_values(["hours", "start"]).iloc[-1]

fact(2, "highest 24 h mean residual load (MWh)", round(roll24.max()), "smard.csv, rolling 24 rows")
fact(2, "highest 24 h window", f"{peak_start:%Y-%m-%d %H:%M} .. {peak_end:%Y-%m-%d %H:%M}", "smard.csv")
fact(2, "longest run below zero (h)", int(longest["hours"]), "smard.csv")
fact(2, "longest run below zero, start", f"{longest['start']:%Y-%m-%d %H:%M}", "smard.csv")
print(FACTS[-4:])

YLIM = (-25_000, 85_000)


def episode(ax, start, end, title):
    """Demand, wind + solar and residual load around one episode."""
    window = time_series.loc[start - pd.Timedelta("36h"): end + pd.Timedelta("36h")]
    ax.axvspan(start, end, color="#ECEAE4", zorder=0)
    ax.fill_between(window.index, 0, window["renewables"], color=COLORS["renewables"], alpha=0.35, lw=0)
    ax.plot(window.index, window["renewables"], color=COLORS["renewables"], lw=2)
    ax.plot(window.index, window["grid_load"], color=COLORS["grid_load"], lw=2.4)
    ax.plot(window.index, window["residual_load"], color=COLORS["residual_load"], lw=3.4)
    ax.axhline(0, color=MUTED, lw=1)
    ax.set_ylim(*YLIM)
    ax.set_title(title, loc="left", fontsize=22, color=INK, pad=10)
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
    thousands(ax)
    return window


fig, axes = new_fig(1664, 600, ncols=2, sharey=True)
w1 = episode(axes[0], peak_start, peak_end, "Winter 2019: little wind, no sun")
w2 = episode(axes[1], longest["start"], longest["end"], f"Summer {longest['start']:%Y}: more wind + solar than demand")
axes[0].set_ylabel("MWh per hour")
# Direct labels, placed by eye on the first panel only
t0 = w1.index[0]
axes[0].text(t0 + pd.Timedelta("78h"), 71_000, "Demand", color=COLORS["grid_load"], fontsize=20, weight="bold")
axes[0].text(t0 + pd.Timedelta("2h"), 33_000, "Residual load", color=COLORS["residual_load"], fontsize=20, weight="bold")
axes[0].text(t0 + pd.Timedelta("30h"), 8_000, "Wind + solar", color="#2A7FB0", fontsize=20, weight="bold")
save(fig, "f01_episodes")

### 2.2 What residual load is: one day

Easter Monday 2026: the day of the largest single saving in spec 11 (viz-03 §3.2).

In [ ]:
DAY = pd.Timestamp("2026-04-06")
day = time_series.loc[DAY: DAY + pd.Timedelta("23h")]
x = day["hour"].to_numpy()
load_, ren = day["grid_load"].to_numpy(), day["renewables"].to_numpy()

fig, ax = new_fig(1664, 620)
ax.fill_between(x, ren, load_, where=load_ >= ren, interpolate=True, color="#D9DCE1", lw=0)
ax.fill_between(x, ren, load_, where=load_ < ren, interpolate=True, color=COLORS["low"], alpha=0.35, lw=0)
ax.plot(x, load_, color=COLORS["grid_load"], lw=3)
ax.plot(x, ren, color=COLORS["renewables"], lw=3)
ax.set_xticks(range(0, 24, 3), [f"{h:02d}:00" for h in range(0, 24, 3)])
ax.set_xlim(0, 23)
ax.set_ylim(0, 62_000)
ax.set_ylabel("MWh per hour")
thousands(ax)
ax.text(1, load_[1] + 2_500, "Demand (grid load)", color=COLORS["grid_load"], fontsize=22, weight="bold")
ax.text(1, ren[1] - 6_000, "Wind + solar", color="#2A7FB0", fontsize=22, weight="bold")
ax.text(21, (load_[21] + ren[21]) / 2, "Gap = residual load:\npower plants, storage\nand imports cover it",
        ha="center", va="center", fontsize=19, color=INK)
mid = int(np.argmax(ren - load_))
ax.annotate("Below zero: surplus must be\nexported, stored or curtailed", xy=(x[mid] + 1, (load_[mid + 1] + ren[mid + 1]) / 2),
            xytext=(x[mid] + 1, 58_000), ha="center", va="center", fontsize=19, color=INK,
            arrowprops={"arrowstyle": "-", "color": MUTED, "lw": 1.4})
fact(3, "Easter Monday 2026 min residual load (MWh)", round(day["residual_load"].min()), "smard.csv")
fact(3, "Easter Monday 2026 hours below zero", int((day["residual_load"] < 0).sum()), "smard.csv")
save(fig, "f02_residual_day")

### 2.3 The midday dip deepens

Mean residual load per local hour over the same calendar window in the first and the latest year
(team-EDA §5.6; the window ends at the record's last day).

In [ ]:
last = time_series.index.max()
in_window = (time_series.index.month < last.month) | (
    (time_series.index.month == last.month) & (time_series.index.day <= last.day)
)
first_year, latest_year = YEARS[0], YEARS[-1]
profile = (
    time_series[in_window & time_series["year"].isin([first_year, latest_year])]
    .groupby(["year", "hour"])["residual_load"].mean().unstack("year")
)
dip = profile[latest_year] - profile[first_year]

fig, ax = new_fig(1100, 620)
ax.fill_between(profile.index, profile[latest_year], profile[first_year], color=COLORS["solar"], alpha=0.25, lw=0)
ax.plot(profile.index, profile[first_year], color="#9AA3AE", lw=3)
ax.plot(profile.index, profile[latest_year], color=COLORS["residual_load"], lw=3.4)
ax.axhline(0, color=MUTED, lw=1)
ax.set_xticks(range(0, 24, 3), [f"{h:02d}:00" for h in range(0, 24, 3)])
ax.set_xlim(0, 23)
ax.set_ylim(0, 50_000)
ax.set_yticks(range(10_000, 50_001, 10_000))
ax.set_ylabel("Mean residual load, MWh per hour")
thousands(ax)
ax.text(0.3, profile[first_year].iloc[0] + 2_200, f"{first_year}", color="#7A838E", fontsize=22, weight="bold")
ax.text(0.3, profile[latest_year].iloc[0] - 4_200, f"{latest_year}", color=INK, fontsize=22, weight="bold")
noon = dip.idxmin()
ax.annotate(f"−{-dip.min():,.0f} MWh\nat {noon:02d}:00", xy=(noon, (profile.loc[noon].sum()) / 2),
            ha="center", va="center", fontsize=20, color=INK)
fact(4, "matched window", f"01 Jan .. {last:%d %b}", "smard.csv")
fact(4, f"midday dip {latest_year} vs {first_year} (MWh)", round(dip.min()), "smard.csv")
save(fig, "f03_midday_dip")

### 2.4 Days with at least one hour below zero

In [ ]:
neg_days = (
    time_series.assign(neg=residual < 0).groupby(["year", "date"])["neg"].any()
    .groupby("year").sum().reindex(YEARS, fill_value=0)
)
fig, ax = new_fig(540, 620)
colors = [COLORS["low"]] * len(YEARS)
bars = ax.bar([str(y) for y in YEARS], neg_days.values, color=colors, width=0.7)
bars[-1].set_alpha(0.55)
bars[-1].set_hatch("//")
bars[-1].set_edgecolor(BG)
for bar, value in zip(bars, neg_days.values):
    ax.text(bar.get_x() + bar.get_width() / 2, value + 2, f"{value}", ha="center", va="bottom", fontsize=18, color=INK)
ax.text(len(YEARS) - 1, neg_days.iloc[-1] + 14, f"to {last:%d %b}", ha="center", fontsize=16, color=MUTED)
ax.set_xticks(range(len(YEARS)), [f"'{str(y)[2:]}" for y in YEARS])
ax.set_yticks([])
ax.spines["bottom"].set_color(LIGHT)
ax.grid(False)
for year, value in neg_days.items():
    fact(4, f"days with an hour below zero, {year}", int(value), "smard.csv")
save(fig, "f04_negative_days")

### 2.5 Risk days of the test year

One bar per day from its lowest to its highest hour; the high threshold is the rolling P99 of
`risk-definition.ipynb`, the low edge is zero (both evaluated in spec 10).

In [ ]:
test_days = risk[pd.to_datetime(risk["date"]).between(TEST_HOURS.min().normalize(), TEST_HOURS.max())].copy()
test_days["date"] = pd.to_datetime(test_days["date"])
is_high = test_days["high_risk_rolling_any"] == True  # noqa: E712 (empty = not evaluable)
is_low = test_days["low_risk_zero_any"] == True  # noqa: E712
quiet = ~is_high & ~is_low

fig, ax = new_fig(1664, 600)
for mask, color in [(quiet, "#D9DCE1"), (is_high, COLORS["high"]), (is_low, COLORS["low"])]:
    part = test_days[mask]
    ax.vlines(part["date"], part["residual_min"], part["residual_max"], color=color, lw=2.6)
ax.step(test_days["date"], test_days["high_threshold_rolling"], where="mid", color=COLORS["high"], lw=2, ls="--")
ax.axhline(0, color=COLORS["low"], lw=2, ls="--")
ax.set_ylabel("MWh per hour")
thousands(ax)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
ax.set_xlim(test_days["date"].min() - pd.Timedelta("2D"), test_days["date"].max() + pd.Timedelta("2D"))
ax.set_ylim(-18_000, 80_000)
ax.text(test_days["date"].iloc[5], 72_500,
        f"High-risk line: top 1 % of the past 365 days  ·  {is_high.sum()} days crossed it",
        color=COLORS["high"], fontsize=19, weight="bold")
ax.text(test_days["date"].iloc[5], -9_500, f"Zero line  ·  {is_low.sum()} days dropped below it",
        color=COLORS["low"], fontsize=19, weight="bold")
fact(5, "test-year high-risk days (rolling P99, any)", int(is_high.sum()), "risk_labels_daily.csv")
fact(5, "test-year below-zero days (zero, any)", int(is_low.sum()), "risk_labels_daily.csv")
save(fig, "f05_risk_days")

## 3 How and what: forecasts
### 3.1 A typical week

The full ISO week whose headline-model skill is closest to its test-year skill, so the example is
neither a lucky nor an unlucky week.

In [ ]:
weekly = abs_err.groupby(abs_err.index.to_period("W-SUN")).agg(["mean", "size"])
weekly = weekly[weekly[("model", "size")] >= 167]
weekly_skill = 100 * (1 - weekly[("model", "mean")] / weekly[("smard", "mean")])
week = (weekly_skill - common).abs().idxmin()
w_start, w_end = week.start_time, week.end_time.floor("h")
fact(6, "example week", f"{w_start:%Y-%m-%d} .. {w_end:%Y-%m-%d}", "rule: weekly skill closest to the year's")
fact(6, "example week skill (%)", round(weekly_skill[week], 1), "model_forecast_errors_hourly.csv")
fact(10, "weeks the headline model beats SMARD", f"{(weekly_skill > 0).sum()} of {len(weekly_skill)}", "model_forecast_errors_hourly.csv")

wk = pd.DataFrame({
    "actual": smard.loc[w_start:w_end, "residual_load"],
    "smard": smard.loc[w_start:w_end, "fc_residual_load"],
    "model": head.loc[w_start:w_end, "forecast"],
})
WEEK_YLIM = (min(wk.min()) - 4_000, max(wk.max()) + 9_000)


def week_plot(with_model):
    """Actual vs SMARD (and the headline model) over the example week."""
    fig, ax = new_fig(1664, 600)
    ax.fill_between(wk.index, wk["actual"], wk["smard"], color="#E4E1DA", lw=0, label="SMARD's error")
    ax.plot(wk.index, wk["actual"], color=MODEL_STYLE["actual"]["color"], lw=3.4, label="Actual residual load")
    ax.plot(wk.index, wk["smard"], color=MODEL_STYLE["smard"]["color"], lw=2.4, ls="--", label="SMARD forecast")
    if with_model:
        ax.plot(wk.index, wk["model"], color=MODEL_STYLE[HEADLINE[0]]["color"], lw=3,
                label="Our forecast (random forest hybrid)")
    ax.axhline(0, color=MUTED, lw=1)
    ax.set_ylim(*WEEK_YLIM)
    ax.set_ylabel("MWh per hour")
    thousands(ax)
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b"))
    ax.legend(loc="upper left", ncols=4, fontsize=18, handlelength=1.6, bbox_to_anchor=(0, 1.02))
    return fig


save(week_plot(False), "f06_week_smard")
save(week_plot(True), "f07_week_model")

### 3.2 Every model against SMARD

In [ ]:
ROWS = [SMARD_ROW] + sorted(
    [(m, "rolling") for m in MODEL_STYLE if m not in ("smard", "actual", ENSEMBLE[0])],
    key=lambda r: -VALUE.loc[r, "MAE"],
) + [ENSEMBLE]
mae = [VALUE.loc[r, "MAE"] for r in ROWS]
colors = [MODEL_STYLE["smard"]["color"] if r == SMARD_ROW
          else MODEL_STYLE[r[0]]["color"] if r == HEADLINE else LIGHT for r in ROWS]

fig, ax = new_fig(1100, 620)
y = np.arange(len(ROWS))[::-1].astype(float)
y[0] += 0.4   # SMARD apart at the top
y[-1] -= 0.4  # ensemble apart at the bottom
ax.barh(y, mae, color=colors, height=0.72)
for yi, r, value in zip(y, ROWS, mae):
    label = MODEL_STYLE[r[0]]["label"]
    weight = "bold" if r in (HEADLINE, SMARD_ROW) else "normal"
    ax.text(-60, yi, label, ha="right", va="center", fontsize=19, color=INK, weight=weight)
    note = "" if r == SMARD_ROW else f"   −{VALUE.loc[r, 'skill_pct']:.1f} %"
    ax.text(value + 40, yi, f"{value:,.0f}{note}", va="center", fontsize=19, color=INK, weight=weight)
ax.set_yticks([])
ax.set_xlim(0, 3_700)
ax.set_xlabel("Mean absolute error, MWh per hour (test year)")
ax.grid(axis="x", color=GRID)
ax.grid(axis="y", visible=False)
thousands(ax, "x")
for r in ROWS:
    fact(9, f"MAE {r[0]} {r[1]}", round(VALUE.loc[r, "MAE"]), "model/ensemble_scoreboard.csv")
    fact(9, f"skill {r[0]} {r[1]} (%)", round(VALUE.loc[r, "skill_pct"], 1), "model/ensemble_scoreboard.csv")
save(fig, "f08_models")

In [ ]:
# Same scoreboard flipped: SMARD at the bottom, skill as a positive improvement
bar_colors = [MODEL_STYLE["smard"]["color"] if r == SMARD_ROW
              else MODEL_STYLE[r[0]]["color"] if r == HEADLINE else LIGHT for r in ROWS]
fig, ax = new_fig(1100, 620)
y = np.arange(len(ROWS)).astype(float)
y[0] -= 0.4   # SMARD apart at the bottom
y[-1] += 0.4  # ensemble apart at the top
ax.barh(y, mae, color=bar_colors, height=0.72)
for yi, r, value in zip(y, ROWS, mae):
    label = MODEL_STYLE[r[0]]["label"]
    weight = "bold" if r in (HEADLINE, SMARD_ROW) else "normal"
    ax.text(-60, yi, label, ha="right", va="center", fontsize=19, color=INK, weight=weight)
    note = "" if r == SMARD_ROW else f"   +{VALUE.loc[r, 'skill_pct']:.1f} %"
    ax.text(value + 40, yi, f"{value:,.0f}{note}", va="center", fontsize=19, color=INK, weight=weight)
ax.set_yticks([])
ax.set_xlim(0, 3_700)
ax.set_xlabel("Mean absolute error, MWh per hour (test year)")
ax.grid(axis="x", color=GRID)
ax.grid(axis="y", visible=False)
thousands(ax, "x")
save(fig, "f08_models_flipped")

### 3.3 Month by month

In [ ]:
monthly = abs_err.groupby(abs_err.index.to_period("M")).agg(["mean", "size"])
full = monthly[("model", "size")] >= monthly.index.days_in_month * 24 - 1
monthly = monthly[full]
monthly_skill = 100 * (1 - monthly[("model", "mean")] / monthly[("smard", "mean")])

fig, ax = new_fig(1100, 560)
colors = [MODEL_STYLE[HEADLINE[0]]["color"] if v > 0 else "#9AA3AE" for v in monthly_skill]
ax.bar(range(len(monthly_skill)), monthly_skill.values, color=colors, width=0.72)
ax.axhline(0, color=MUTED, lw=1.2)
ax.set_xticks(range(len(monthly_skill)), [p.strftime("%b\n%Y") if p.month == 1 or i == 0 else p.strftime("%b")
                                          for i, p in enumerate(monthly_skill.index)])
for i, value in enumerate(monthly_skill.values):
    if value <= 0 or value == monthly_skill.max():
        ax.text(i, value + (1.2 if value > 0 else -1.2), f"{value:+.1f} %".replace("-", "−"), ha="center",
                va="bottom" if value > 0 else "top", fontsize=18, color=INK)
ax.set_ylabel("Error reduction vs SMARD, %")
ax.set_ylim(-8, 42)
for period, value in monthly_skill.items():
    fact(10, f"monthly skill {period}", round(value, 1), "model_forecast_errors_hourly.csv")
fact(10, "full months beating SMARD", f"{(monthly_skill > 0).sum()} of {len(monthly_skill)}", "model_forecast_errors_hourly.csv")
first_half = abs_err.loc["2026-01":"2026-06"]
gain = abs_err["smard"] - abs_err["model"]
fact(14, "share of the year's gain from Jan-Jun 2026 (%)", round(100 * gain.loc["2026-01":"2026-06"].sum() / gain.sum()), "model_forecast_errors_hourly.csv")
save(fig, "f09_monthly")

### 3.4 Extremes

MAE by actual level; per bin the spec 09 rank-1 pick, "all hours" the headline model.

In [ ]:
BINS = {
    "overall": ("All hours", "MAE"),
    "ordinary": ("Ordinary\n(middle 50 %)", "MAE_ordinary_by_actual"),
    "below_zero": ("Below zero", "MAE_below_zero_by_actual"),
    "low_extreme": ("Lowest 1 %", "MAE_low_extreme_by_actual"),
    "high_extreme": ("Highest 1 %", "MAE_high_extreme_by_actual"),
}
fig, ax = new_fig(1664, 600)
width = 0.36
for i, (key, (label, metric)) in enumerate(BINS.items()):
    pick = (BIN_PICKS[key], "rolling")
    base, ours = VALUE.loc[SMARD_ROW, metric], VALUE.loc[pick, metric]
    ax.bar(i - width / 2 - 0.02, base, width, color=MODEL_STYLE["smard"]["color"])
    ax.bar(i + width / 2 + 0.02, ours, width, color=MODEL_STYLE[pick[0]]["color"])
    ax.text(i + width / 2 + 0.02, ours + 80, f"−{100 * (1 - ours / base):.0f} %", ha="center", va="bottom",
            fontsize=20, color=INK, weight="bold")
    hours = int(COUNT.loc[pick, metric])
    ax.text(i, -420, f"{label}", ha="center", va="top", fontsize=19, color=INK, weight="bold")
    ax.text(i, -1_350, f"{hours:,} h · {MODEL_STYLE[pick[0]]['label']}", ha="center", va="top", fontsize=16, color=MUTED)
    fact(12, f"{key} SMARD MAE", round(base), "model_scoreboard.csv")
    fact(12, f"{key} {pick[0]} MAE", round(ours), "model_scoreboard.csv")
    fact(12, f"{key} hours", hours, "model_scoreboard.csv")
ax.set_xticks([])
ax.set_ylim(0, 4_800)
ax.set_ylabel("MAE, MWh per hour")
thousands(ax)
first = VALUE.loc[SMARD_ROW, "MAE"]
ax.text(-width / 2 - 0.02, first + 80, "SMARD", ha="center", va="bottom", fontsize=19, color=MUTED, weight="bold")
save(fig, "f10_extremes")

In [ ]:
# Same bins with the ensemble pick (spec 09) as a third, hatched bar per group
fig, ax = new_fig(1664, 600)
width = 0.27
ens_style = MODEL_STYLE[ENSEMBLE[0]]


def change(ours, base):
    """Error change vs SMARD as a label, e.g. −12 %."""
    return f"{100 * (ours / base - 1):+.0f} %".replace("-", "−")


with plt.rc_context({"hatch.linewidth": 1.6, "hatch.color": ens_style["color"]}):
    for i, (key, (label, metric)) in enumerate(BINS.items()):
        pick = (BIN_PICKS[key], "rolling")
        base, ours, ens = (VALUE.loc[row, metric] for row in (SMARD_ROW, pick, ENSEMBLE))
        hours = int(COUNT.loc[pick, metric])
        assert int(COUNT.loc[ENSEMBLE, metric]) == hours, f"{key}: ensemble and pick count different hours"
        ax.bar(i - width - 0.03, base, width, color=MODEL_STYLE["smard"]["color"])
        ax.bar(i, ours, width, color=MODEL_STYLE[pick[0]]["color"])
        ax.bar(i + width + 0.03, ens, width, facecolor=BG, edgecolor=ens_style["color"], hatch="//",
               linestyle=":", linewidth=2.4)
        for x, value in [(i, ours), (i + width + 0.03, ens)]:
            ax.text(x, value + 80, change(value, base), ha="center", va="bottom", fontsize=18, color=INK, weight="bold")
        ax.text(i, -420, f"{label}", ha="center", va="top", fontsize=19, color=INK, weight="bold")
        ax.text(i, -1_350, f"{hours:,} h · {MODEL_STYLE[pick[0]]['label']}", ha="center", va="top", fontsize=16, color=MUTED)
        fact(12, f"{key} {ENSEMBLE[0]} MAE", round(ens), "ensemble_scoreboard.csv")
ax.set_xticks([])
ax.set_ylim(0, 4_800)
ax.set_ylabel("MAE, MWh per hour")
thousands(ax)
first, first_ens = VALUE.loc[SMARD_ROW, "MAE"], VALUE.loc[ENSEMBLE, "MAE"]
ax.text(-width - 0.03, first + 80, "SMARD", ha="center", va="bottom", fontsize=19, color=MUTED, weight="bold")
ax.text(width + 0.03, first_ens + 520, "Ensemble", ha="center", va="bottom", fontsize=19, color=ens_style["color"],
        weight="bold")
save(fig, "f10_extremes_ensemble")

### 3.5 Risk-day scorecard (numbers for the slide tiles)

Days of the test year; high = rolling P99 `any`, low = zero `any`, spec 10 picks.

In [ ]:
def outcomes(source, direction, basis):
    """Hits, misses and false alarms of `source` (model / smard) against the actual flag."""
    actual = model_risk[f"actual_{direction}_risk_{basis}_any"] == True  # noqa: E712
    flagged = model_risk[f"{source}_{direction}_risk_{basis}_any"] == True  # noqa: E712
    return {"actual": int(actual.sum()), "hits": int((actual & flagged).sum()),
            "misses": int((actual & ~flagged).sum()), "false_alarms": int((~actual & flagged).sum())}


for direction, basis in [("high", "rolling"), ("low", "zero")]:
    for source in ["model", "smard"]:
        result = outcomes(source, direction, basis)
        name = RISK_PICKS[direction] if source == "model" else "smard"
        print(direction, basis, name, result)
        for k, v in result.items():
            fact(13, f"{direction} {basis} {name} {k}", v, "model_risk_labels_daily.csv")

### 3.6 What the gain could be worth (reBAP proxy, spec 11)

In [ ]:
fig, ax = new_fig(1664, 600)
ends = {}
for model in [ENSEMBLE[0], HEADLINE[0]]:
    rows = cost.query("model == @model and split_method == 'rolling'").set_index("timestamp").sort_index()
    cum = rows["saved_eur"].cumsum() / 1e6
    ax.plot(cum.index, cum.values, color=MODEL_STYLE[model]["color"], lw=3.2 if model == ENSEMBLE[0] else 2.6)
    ends[model] = cum
    fact(14, f"cumulative saving {model} (M EUR)", round(cum.iloc[-1], 1), "model_rebap_cost_hourly.csv")
ax.axhline(0, color=MUTED, lw=1.2, ls="--")
ax.text(TEST_HOURS.max() + pd.Timedelta("4D"), 0, "SMARD = 0", color=MUTED, fontsize=18, va="bottom")
for model, label in [(ENSEMBLE[0], "Ensemble (team pick)"), (HEADLINE[0], "Random forest hybrid")]:
    cum = ends[model]
    ax.text(cum.index[-1] + pd.Timedelta("4D"), cum.iloc[-1], f"{label}\n{cum.iloc[-1]:,.0f} M€",
            color=MODEL_STYLE[model]["color"], fontsize=19, weight="bold", va="center")
easter = ends[ENSEMBLE[0]]
jump = easter.loc["2026-04-06"].iloc[-1] - easter.loc[:"2026-04-05"].iloc[-1]
ax.annotate(f"Easter Monday:\n+{jump:,.0f} M€ in one day", xy=(pd.Timestamp("2026-04-06 15:00"), easter.loc["2026-04-06 15:00"]),
            xytext=(pd.Timestamp("2026-01-20"), 300), fontsize=19, color=INK,
            arrowprops={"arrowstyle": "-", "color": MUTED, "lw": 1.4})
aug = cost.query("model == @ENSEMBLE[0] and split_method == 'rolling'").set_index("timestamp").loc["2026-08", "saved_eur"].sum() / 1e6
ax.annotate(f"August: −{-aug:,.0f} M€", xy=(pd.Timestamp("2026-08-25"), easter.loc["2026-08-25"].iloc[-1]),
            xytext=(pd.Timestamp("2026-06-20"), 250), fontsize=19, color=INK,
            arrowprops={"arrowstyle": "-", "color": MUTED, "lw": 1.4})
fact(14, "Easter Monday jump, ensemble (M EUR)", round(jump, 1), "model_rebap_cost_hourly.csv")
fact(14, "August 2026 saving, ensemble (M EUR)", round(aug, 1), "model_rebap_cost_hourly.csv")
smard_cost = cost.query("model == 'smard'")["cost_eur"].sum() / 1e6
fact(14, "SMARD cost proxy (M EUR)", round(smard_cost, 1), "model_rebap_cost_hourly.csv")
ax.set_ylabel("Cumulative saving vs SMARD, M€")
ax.set_xlim(TEST_HOURS.min(), TEST_HOURS.max() + pd.Timedelta("62D"))
ax.set_xticks(pd.date_range(TEST_HOURS.min().normalize() + pd.offsets.MonthBegin(1), TEST_HOURS.max(), freq="2MS"))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.grid(axis="x", visible=False)
save(fig, "f11_money")

## 4 Slide numbers

Every number the slides quote, with its source; check these after any re-run.

In [ ]:
facts = pd.DataFrame(FACTS)
if EXPORT_ENABLED:
    facts.to_csv(DATA_DIR.parent / "presentation" / "slide_numbers.csv", index=False)
pd.set_option("display.max_rows", 200)
print(facts.to_string())